# Lesson 10 · Files and errors

Real programs read and write files and deal with things that go wrong. You will learn to handle errors on purpose instead of letting your program crash.

**What you will practice**

- Catch errors with `try` / `except`
- Raise your own errors
- Read and write text files with `with open(...)`
- Write functions that cope with bad input and missing files

Run the setup cell below first. In every exercise: read the *Quick review*, write your answer in the code cell, then run the check cell.

In [ ]:
# Run this cell first. It defines the helpers used to check your answers.
import contextlib, io


def check(condition, message="Not quite. Re-read the task and try again."):
    assert condition, message
    print("Correct!")


def check_prediction(prediction, code):
    def clean(text):
        return "\n".join(line.rstrip() for line in str(text).strip().splitlines())

    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        exec(code, {})
    assert clean(prediction) == clean(buffer.getvalue()), (
        "Not quite. Run the code cell above, compare its output with your prediction, and try again."
    )
    print("Correct!")

## Warm-up: what do you remember?

Try to answer from memory before opening the answers.

**1.** How do you read a dictionary key without risking a `KeyError`?

<details><summary>Answer</summary>

`d.get(key, default)`.

</details>

**2.** What is `{1, 2, 2, 3}`?

<details><summary>Answer</summary>

A set with three items: `{1, 2, 3}`.

</details>

**3.** What does `d.items()` give?

<details><summary>Answer</summary>

(key, value) pairs.

</details>

## Exercises

### Exercise 1: try and except

`Predict`

**Quick review**

Code in `try` runs normally. If it raises an error of the type named in `except`, Python jumps to the `except` block instead of crashing. `int("abc")` raises `ValueError`.

**Task**

Predict the output.

Read this code and write your prediction **before** running anything:

```python
def to_number(text):
    try:
        return int(text)
    except ValueError:
        return -1

print(to_number("42"))
print(to_number("forty-two"))
print(to_number(""))
```

In [ ]:
prediction = """
write here what you think will be printed
"""

Now run the code to see what really happens, then run the check cell below.

In [ ]:
def to_number(text):
    try:
        return int(text)
    except ValueError:
        return -1

print(to_number("42"))
print(to_number("forty-two"))
print(to_number(""))

In [ ]:
check_prediction(prediction, """
def to_number(text):
    try:
        return int(text)
    except ValueError:
        return -1

print(to_number("42"))
print(to_number("forty-two"))
print(to_number(""))
""")

### Exercise 2: Safe division

`Make`

**Quick review**

Catch only the error you expect (`ZeroDivisionError` here). Catching everything hides real bugs.

**Task**

Write `safe_divide(a, b)` that returns `a / b`, or `None` if `b` is zero.

In [ ]:
def safe_divide(a, b):
    ...

In [ ]:
check(safe_divide(10, 4) == 2.5 and safe_divide(1, 0) is None and safe_divide(0, 5) == 0.0, "Test a normal division, a division by zero and zero divided by a number.")

### Exercise 3: Validate an age

`Make`

**Quick review**

Combine `try`/`except` with normal conditions. Return `None` for anything invalid.

**Task**

Write `parse_age(text)` that returns the age as an `int` if `text` is a whole number from 0 to 120, and `None` otherwise.

In [ ]:
def parse_age(text):
    ...

In [ ]:
check(parse_age("35") == 35 and parse_age("abc") is None and parse_age("-5") is None and parse_age("200") is None
      and parse_age("0") == 0 and parse_age("120") == 120, "Test valid ages, text, negative and too large numbers, and both limits.")

### Exercise 4: Write and read a file

`Make`

**Quick review**

`with open(path, "w") as f:` opens a file for writing and closes it for you afterwards. `f.write(text)` writes. Use `"r"` (or nothing) to read, and loop over `f` to get one line at a time. Lines end with a newline, which `line.rstrip("\n")` removes. Add `"\n"` yourself when writing.

**Task**

Write `write_lines(path, lines)` that writes each item of `lines` on its own line, and `read_lines(path)` that returns the lines as a list of strings without the newline.

In [ ]:
def write_lines(path, lines):
    ...


def read_lines(path):
    ...

In [ ]:
import os, tempfile
path = os.path.join(tempfile.gettempdir(), "python_by_practice_lines.txt")
write_lines(path, ["a", "b", "c"])
check(read_lines(path) == ["a", "b", "c"], "What you write should be what you read back.")
write_lines(path, [])
check(read_lines(path) == [], "An empty list should produce an empty file.")
os.remove(path)

### Exercise 5: A missing file is not a crash

`Make`

**Quick review**

Opening a file that does not exist raises `FileNotFoundError`. Handling it lets your program continue with a sensible default.

**Task**

Write `read_or_default(path, default="")` returning the whole text of the file, or `default` if the file does not exist.

In [ ]:
def read_or_default(path, default=""):
    ...

In [ ]:
import os, tempfile
path = os.path.join(tempfile.gettempdir(), "python_by_practice_note.txt")
with open(path, "w") as f:
    f.write("hello")
check(read_or_default(path) == "hello", "An existing file should be read.")
os.remove(path)
check(read_or_default(path) == "" and read_or_default(path, "n/a") == "n/a", "A missing file should give the default.")

### Exercise 6: Count lines and words

`Make`

**Quick review**

Reading a whole file with `f.read()` gives one string; `.splitlines()` and `.split()` cut it into lines and words.

**Task**

Write `file_stats(path)` returning a tuple `(number_of_lines, number_of_words)`.

In [ ]:
def file_stats(path):
    ...

In [ ]:
import os, tempfile
path = os.path.join(tempfile.gettempdir(), "python_by_practice_stats.txt")
with open(path, "w") as f:
    f.write("hello world\nfoo\n")
check(file_stats(path) == (2, 3), "Two lines and three words.")
with open(path, "w") as f:
    f.write("")
check(file_stats(path) == (0, 0), "An empty file has no lines and no words.")
os.remove(path)

### Exercise 7: Parse messy data

`Challenge`

**Quick review**

Real files contain bad lines. Skip the ones you cannot understand instead of crashing. `line.split(",")` cuts a line at the commas.

**Task**

Write `read_scores(path)`. Each line should look like `name,score`. Return a dictionary `{name: score}` with the score as an `int`, and **skip** lines that do not have exactly two parts or whose score is not a whole number.

<details><summary>Hint</summary>

Split each line at the commas. Skip it if you do not get exactly two parts, or if `int()` fails on the score.

</details>

In [ ]:
def read_scores(path):
    ...

In [ ]:
import os, tempfile
path = os.path.join(tempfile.gettempdir(), "python_by_practice_scores.txt")
with open(path, "w") as f:
    f.write("ada,90\nlinus,75\noops\ngrace,abc\n,\nhopper,100,extra\n")
result = read_scores(path)
os.remove(path)
check(result == {"ada": 90, "linus": 75}, "Keep only the valid lines: ada,90 and linus,75.")

## Recap

- `try` / `except SomeError` handles expected problems; catch specific errors.
- `with open(path) as f:` closes the file for you.
- Mode `"w"` overwrites, `"r"` reads; lines end with a newline.
- Skipping bad input with `continue` keeps the program alive.

### Common mistakes

- Catching every error with a bare `except:` and hiding bugs.
- Forgetting `"\n"` when writing lines.
- Opening a file with `"w"` and erasing it by accident.

**Next:** The Level 1 project: a command-line task manager.